In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

mask = ~metadata_df['g207 treatment'].str.strip().str.lower().str.startswith('pre-')
metadata_df = metadata_df[mask]

print(metadata_df.shape)
metadata_df


(6, 10)


,sample_title,patient_id,tissue,diagnosis,glioblastoma subtype,brain location,g207 treatment,age (years),Sex,survival post g207 (days)
samples,,,,,,,,,,
GSM4956152,Patient_1_post_G207,Patient_101,brain tumor,glioblastoma,Mesenchymal,left frontal,post-G207 inoculation,52,male,113
GSM4956153,Patient_3_post_G207,Patient_103,brain tumor,glioblastoma,Classical/Proneural,left frontoparietal,post-G207 inoculation,57,female,233
GSM4956154,Patient_5_post_G207,Patient_105,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,39,female,263
GSM4956155,Patient_6_post_G207,Patient_106,brain tumor,glioblastoma,Mesenchymal,right frontoparietal,post-G207 inoculation,47,female,165
GSM4956156,Patient_7_post_G207,Patient_107,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,66,female,623
GSM4956157,Patient_8_post_G207,Patient_108,brain tumor,glioblastoma,Proneural/Neural,right frontal,post-G207 inoculation,66,male,60


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_97183/3050319327.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,patient_id,tissue,diagnosis,glioblastoma subtype,brain location,g207 treatment,age (years),Sex,survival post g207 (days),...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM4956152,Patient_1_post_G207,Patient_101,brain tumor,glioblastoma,Mesenchymal,left frontal,post-G207 inoculation,52,male,113,...,147371,123,148,148,71555,24394,1746,62896,86,570
GSM4956153,Patient_3_post_G207,Patient_103,brain tumor,glioblastoma,Classical/Proneural,left frontoparietal,post-G207 inoculation,57,female,233,...,139979,181,194,221,72636,21382,1448,44827,113,341
GSM4956154,Patient_5_post_G207,Patient_105,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,39,female,263,...,119778,139,150,157,73095,24103,2033,42283,168,439
GSM4956155,Patient_6_post_G207,Patient_106,brain tumor,glioblastoma,Mesenchymal,right frontoparietal,post-G207 inoculation,47,female,165,...,286311,207,229,241,128545,40605,2701,145684,109,614
GSM4956156,Patient_7_post_G207,Patient_107,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,66,female,623,...,203785,167,169,184,88141,33477,2222,84968,188,691
GSM4956157,Patient_8_post_G207,Patient_108,brain tumor,glioblastoma,Proneural/Neural,right frontal,post-G207 inoculation,66,male,60,...,258024,306,338,485,149121,45366,3608,125984,268,857


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
